In [2]:
# If you run notebook from notebooks/, the repo root is one level up.
import os, sys
sys.path.append(os.path.abspath(".."))

PDF_PATH = "../data/prudentservices_dataset_rag.pdf"   # <-- confirm your filename
print("exists?", os.path.exists(PDF_PATH), PDF_PATH)

exists? True ../data/prudentservices_dataset_rag.pdf


In [3]:
from rag.pdf_ingest import load_or_build_chunks

chunks, cache_path = load_or_build_chunks(PDF_PATH, chunk_size=1400, overlap=200)
print("chunks:", len(chunks))
print("cache:", cache_path)
print("first:", chunks[0].chunk_id, chunks[0].meta)
print(chunks[0].text[:200])

chunks: 60
cache: data/cache\chunks_de39a2e621ecc77e.jsonl
first: c000000 {'source': 'prudentservices_dataset_rag.pdf', 'page': 1, 'char_start': 0, 'char_end': 674}
Prudent Services Dataset Pack 
Public Website Reference + Internal Document Templates 
Disclosure 
Part I is a structured reference that paraphrases publicly available information from 
prudentservice


In [4]:
from rag.index_bm25 import BM25Index

bm25 = BM25Index()
bm25.build(chunks)

hits = bm25.search("training topics", k=5)
for h in hits:
    print(h.chunk_id, round(h.score, 4), "page", h.meta.get("page"))

c000000 5.8771 page 1
c000001 3.349 page 2
c000032 3.2005 page 31
c000025 3.0266 page 24
c000037 2.8829 page 36


In [5]:
from rag.index_dense import DenseIndex

dense = DenseIndex(embed_model="nomic-embed-text")
dense.build(chunks, batch_size=32, use_cache=True)

hits = dense.search("fire prevention and emergency evacuation training", k=5)
for h in hits:
    print(h.chunk_id, round(h.score, 4), "page", h.meta.get("page"))

c000029 0.7346 page 28
c000032 0.7048 page 31
c000028 0.6346 page 27
c000001 0.5917 page 2
c000036 0.5854 page 35


In [7]:
import os, sys
sys.path.append(os.path.abspath("."))  # if notebook in repo root

from rag.pdf_ingest import load_or_build_chunks
from rag.index_bm25 import BM25Index
from rag.index_dense import DenseIndex
from rag.hybrid_rank import hybrid_rank

PDF_PATH = "../data/prudentservices_dataset_rag.pdf"  # update if different
chunks, _ = load_or_build_chunks(PDF_PATH)

bm25 = BM25Index(); bm25.build(chunks)
dense = DenseIndex(embed_model="nomic-embed-text"); dense.build(chunks)

bm = bm25.search("training topics", k=20)
de = dense.search("fire prevention and emergency evacuation training", k=20)

hy = hybrid_rank(bm, de, x_bm25=0.5, y_dense=0.5, topk=10)
for h in hy[:5]:
    print(h.chunk_id, round(h.score,4), round(h.bm25_norm or 0,3), round(h.dense_norm or 0,3))

c000032 0.7185 0.545 0.892
c000000 0.5794 1.0 0.159
c000001 0.5269 0.57 0.484
c000029 0.5 0 1.0
c000002 0.418 0.451 0.385
